In [1]:
import csv
import os
import sys

import numpy as np
import scope
import scope_sco
from copy import deepcopy

sys.path.insert(0, os.path.abspath('..'))
from atlas_functions import find_optimized_minimum

In [5]:
data_path     = "/Users/sergivela/Documents/SCOPE/ATLAS/Data/"                    # Update with your path
analysis_path = "/Users/sergivela/Documents/SCOPE/Sub-Projects/1-ATLAS/Analysis/"  # Update with your path 
env_path      = "/Users/sergivela/Documents/SCOPE/ATLAS/Data/scope_env_atlas.npy" # Update with your path
outputs_path  = analysis_path + "Outputs/"

# Normalizes the configured strings and defines the analysis output folder.
data_path     = os.path.abspath(os.path.expanduser(data_path)) + os.sep
env_path      = os.path.abspath(os.path.expanduser(env_path))

# Loads Environment
env = scope.load_binary(env_path)

# Output Paths
unique_csv_path     = outputs_path + "unique_systems.csv"
comparison_csv_path = outputs_path + "ho_qrrho_thermodynamic_comparison.csv"
thermo_csv_path     = outputs_path + "unique_thermodynamic_data.csv"
statistics_csv_path = outputs_path + "qrrho_dSvib_denticity_statistics.csv"

print(f"Data directory:    {data_path}")
print(f"SCOPE environment: {env_path}")

Data directory:    /Users/sergivela/Documents/SCOPE/ATLAS/Data/
SCOPE environment: /Users/sergivela/Documents/SCOPE/ATLAS/Data/scope_env_atlas.npy


In [3]:
unique_systems = []
with open(unique_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sys = scope.load_binary(f"{data_path}/{row['path']}")
        sys.set_paths_from_environment(env)
        unique_systems.append(sys)

print("Unique Systems:", len(unique_systems))

Unique Systems: 602


In [7]:
datas = []

for debug_sys in unique_systems:
    HS_ref_mol = debug_sys.find_source("ref_hs_mol")[1]
    LS_ref_mol = debug_sys.find_source("ref_ls_mol")[1]

    # Selects b3lyp_opt minima, falling back to corr_opt when necessary
    HS_found, HS_min = find_optimized_minimum(HS_ref_mol)
    LS_found, LS_min = find_optimized_minimum(LS_ref_mol)

    # Compute HO (default)
    HS_min.get_thermal_data(temp=range(10,501,1), overwrite=True)
    LS_min.get_thermal_data(temp=range(10,501,1), overwrite=True)
    worked, HO_T12 = scope_sco.sco_functions.get_T12(debug_sys.find_branch("isolated")[1], HS_min, LS_min, flexible=True, overwrite=True, debug=0)

    HO_HS_gtot = deepcopy(HS_min.results["Gtot"])
    HO_LS_gtot = deepcopy(LS_min.results["Gtot"])
    HO_HS_svib = deepcopy(HS_min.results["Svib"])
    HO_LS_svib = deepcopy(LS_min.results["Svib"])

    # Compute QRRHO
    vib_options = {"typ": "QRRHO"}

    HS_min.get_thermal_data(temp=range(10,501,1), vib_options=vib_options, overwrite=True)
    LS_min.get_thermal_data(temp=range(10,501,1), vib_options=vib_options, overwrite=True)
    worked, QRRHO_T12 = scope_sco.sco_functions.get_T12(debug_sys.find_branch("isolated")[1], HS_min, LS_min, flexible=True, overwrite=True, debug=0)

    QRRHO_HS_gtot = deepcopy(HS_min.results["Gtot"])
    QRRHO_LS_gtot = deepcopy(LS_min.results["Gtot"])
    QRRHO_HS_svib = deepcopy(HS_min.results["Svib"])
    QRRHO_LS_svib = deepcopy(LS_min.results["Svib"])

    # Compare Gtot
    G_dHO = HO_HS_gtot - HO_LS_gtot
    G_dQRRHO = QRRHO_HS_gtot - QRRHO_LS_gtot
    G_dMethod = G_dHO - G_dQRRHO
    Gdiff = G_dMethod.find_value_with_property("temperature", 298)

    # Compare Gtot
    S_dHO = HO_HS_svib - HO_LS_svib
    S_dQRRHO = QRRHO_HS_svib - QRRHO_LS_svib
    S_dMethod = S_dHO - S_dQRRHO
    Sdiff = S_dMethod.find_value_with_property("temperature", 298)

    datas.append(list([debug_sys.refcode, Gdiff, Sdiff, HO_T12, QRRHO_T12]))

    print(debug_sys.refcode, np.round(Sdiff.value * 298.15 * scope.constants.har2kJ,4), HO_T12, QRRHO_T12)

ABITEM 3.0951 T12: None T12: None
ACAHUJ02 6.7697 T12: None T12: None
ADAQUU 4.4251 T12: 166.00000000 K T12: 204.00000000 K
ADAREF 3.3455 T12: None T12: None
ADEJOK 6.2583 T12: None T12: None
ADEQIL 2.7178 T12: 272.00000000 K T12: 328.00000000 K
ADEQOR 2.7332 T12: 288.00000000 K T12: 347.00000000 K
AFANEB02 8.9247 T12: 101.00000000 K T12: 126.00000000 K
AFOYUS02 3.1192 T12: None T12: None
AGIZEX 4.0561 T12: None T12: None
AHINUD 5.5413 T12: 208.00000000 K T12: 258.00000000 K
AJAXEP 3.1148 T12: None T12: None
AKEMUY 1.6148 T12: None T12: None
AKENAF -2.4583 T12: None T12: None
AKEQEM -0.4289 T12: 351.00000000 K T12: 344.00000000 K
ALIKUC -4.4012 T12: None T12: None
ALILEN 7.0764 T12: 280.00000000 K T12: 400.00000000 K
ALUBOB 4.4135 T12: 177.00000000 K T12: 225.00000000 K
AMIHUA 5.3504 T12: 199.00000000 K T12: 258.00000000 K
APAFEH03 4.2993 T12:  78.00000000 K T12:  85.00000000 K
APINUJ 5.7334 T12: None T12: None
ARADOO 0.8138 T12: None T12: None
ARAGEH01 2.8887 T12: None T12: None
AROZI

In [ ]:
fieldnames = [
    "refcode",
    "DeltaGtot_HO_minus_QRRHO_298[kJ/mol]",
    "DeltaSvib_HO_minus_QRRHO_298[J/mol/K]",
    "T12_HO[K]",
    "T12_QRRHO[K]",
]

with open(comparison_csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for refcode, Gdiff, Sdiff, HO_T12, QRRHO_T12 in datas:
        HO_T12_value = getattr(HO_T12, "value", HO_T12)
        QRRHO_T12_value = getattr(QRRHO_T12, "value", QRRHO_T12)

        writer.writerow({
            "refcode": refcode,
            "DeltaGtot_HO_minus_QRRHO_298[kJ/mol]": Gdiff.value * scope.constants.har2kJmol,
            "DeltaSvib_HO_minus_QRRHO_298[J/mol/K]": Sdiff.value * scope.constants.har2kJmol * 1000,
            "T12_HO[K]": "" if HO_T12_value is None else HO_T12_value,
            "T12_QRRHO[K]": "" if QRRHO_T12_value is None else QRRHO_T12_value,
        })

print(f"Saved {len(datas)} entries to {comparison_csv_path}")

Saved 602 entries to /Users/sergivela/Documents/SCOPE/Sub-Projects/1-ATLAS/Analysis/Outputs/ho_qrrho_thermodynamic_comparison.csv


## Denticity-resolved QRRHO vibrational entropy

The absolute QRRHO $\Delta S_{\mathrm{vib}}$ is recovered from the stored harmonic value and the saved HO$-$QRRHO difference. This avoids repeating the expensive thermochemistry calculation. Statistics are reported separately for each homoleptic denticity and for heteroleptic complexes.

In [6]:
temperature = 298.0

with open(thermo_csv_path, newline="") as f:
    thermo_by_refcode = {row["refcode"]: row for row in csv.DictReader(f)}

qrrho_dSvib_rows = []
with open(comparison_csv_path, newline="") as f:
    for row in csv.DictReader(f):
        refcode = row["refcode"][:6]
        thermo = thermo_by_refcode.get(refcode)
        if thermo is None:
            continue

        dSvib_HO = float(thermo["dSvib·T[kJ/mol](T=298K)"]) * 1000 / temperature
        dSvib_difference = float(row["DeltaSvib_HO_minus_QRRHO_298[J/mol/K]"])
        qrrho_dSvib_rows.append({
            "refcode": refcode,
            "denticity": thermo["denticity"],
            "dSvib_QRRHO[J/mol/K]": dSvib_HO - dSvib_difference,
        })

denticity_groups = [("1", "d = 1"), ("2", "d = 2"), ("3", "d = 3"),
                     ("6", "d = 6"), ("unknown", "Heteroleptic")]
statistics = []

for denticity, group in denticity_groups:
    values = np.array([
        row["dSvib_QRRHO[J/mol/K]"] for row in qrrho_dSvib_rows
        if row["denticity"] == denticity
    ])
    statistics.append({
        "Group": group,
        "Num Entries": len(values),
        "Mean dSvib QRRHO [J/mol/K]": np.mean(values),
        "Median dSvib QRRHO [J/mol/K]": np.median(values),
        "Std dSvib QRRHO [J/mol/K]": np.std(values, ddof=1),
        "Q1 dSvib QRRHO [J/mol/K]": np.percentile(values, 25),
        "Q3 dSvib QRRHO [J/mol/K]": np.percentile(values, 75),
    })

with open(statistics_csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(statistics[0]))
    writer.writeheader()
    writer.writerows(statistics)

print(f"{'Group':<14} {'N':>4} {'Mean':>10} {'Median':>10} {'Std':>10} {'Q1':>10} {'Q3':>10}")
for row in statistics:
    print(
        f"{row['Group']:<14} {row['Num Entries']:>4} "
        f"{row['Mean dSvib QRRHO [J/mol/K]']:>10.2f} "
        f"{row['Median dSvib QRRHO [J/mol/K]']:>10.2f} "
        f"{row['Std dSvib QRRHO [J/mol/K]']:>10.2f} "
        f"{row['Q1 dSvib QRRHO [J/mol/K]']:>10.2f} "
        f"{row['Q3 dSvib QRRHO [J/mol/K]']:>10.2f}"
    )

print(f"\nSaved statistics to {statistics_csv_path}")

Group             N       Mean     Median        Std         Q1         Q3
d = 1            12      65.59      68.29       9.93      58.43      70.39
d = 2            50      44.73      46.16       9.72      39.95      49.16
d = 3           207      42.83      43.79      10.21      39.04      49.29
d = 6            30      37.08      38.80       8.45      33.95      41.79
Heteroleptic    303      40.68      40.31      12.23      33.33      48.51

Saved statistics to /Users/sergivela/Documents/SCOPE/Sub-Projects/1-ATLAS/Analysis/Outputs/qrrho_dSvib_denticity_statistics.csv
